
## Delta **tables**



In [0]:


# Spark Session

spark

In [0]:

spark.conf.get("spark.sql.catalogImplementation")
 

In [0]:
%sql 

show databases

In [0]:
%sh

cd /FileStore/tables/data/input/


In [0]:
df_sales = spark.read.parquet("/FileStore/tables/data/input/userdata.parquet")



In [0]:
display(df_sales)

In [0]:

%sql use catalog hive_metastore



In [0]:
df_sales.write.format("parquet").mode("overwrite").option("path", 'dbfs:/FileStore/tables/data/output/sales_parquet_1').saveAsTable("sales_parquet")

In [0]:
display(dbutils.fs.ls('dbfs:/FileStore/tables/data/output/sales_parquet_1'))

In [0]:
%sql

show tables in default

In [0]:
%sql

desc extended sales_parquet

In [0]:
%sql

select * from sales_parquet set first_name = 'John' where id = '12';

In [0]:
# write the parquet in delta tabel format


df_sales.write.format("delta").mode("overwrite").option("path", 'dbfs:/FileStore/tables/data/output/sales_delta').saveAsTable("sales_delta")


In [0]:
%sql 

show tables in default

In [0]:
display(dbutils.fs.ls('dbfs:/FileStore/tables/data/output/sales_delta/_delta_log/00000000000000000000.json'))

In [0]:
%sql 

desc extended sales_delta

In [0]:
dbutils.fs.head("dbfs:/FileStore/tables/data/output/sales_delta/_delta_log/00000000000000000000.json")

In [0]:
%sql

desc history sales_delta

In [0]:
%sql

update default.sales_delta set first_name = 'John' where id = '12';

In [0]:
%sql 

select * from sales_delta where id = '12'


In [0]:
%sql

desc history sales_delta

In [0]:
display(dbutils.fs.ls('dbfs:/FileStore/tables/data/output/sales_delta/_delta_log'))

In [0]:
# read the delta  using pyspark api

df_sales_delta = spark.read.format("delta").load("dbfs:/FileStore/tables/data/output/sales_delta")
display(df_sales_delta)

In [0]:
df_sales_delta = spark.read.format("delta").load("dbfs:/FileStore/tables/data/output/sales_delta")
display(df_sales_delta.where("id = 12"))

In [0]:
%sql

select * from sales_delta


In [0]:

df_new = spark.sql("select *, current_timestamp() as time_now from sales_delta where id = '12'")

display(df_new)



In [0]:
df_new.write.format("delta").mode("append").option("mergeSchema", "true").option("path", 'dbfs:/FileStore/tables/data/output/sales_delta').saveAsTable("sales_delta")

In [0]:
display(df_new)

In [0]:
%sql

select * from sales_delta

In [0]:
%sql

DESCRIBE HISTORY sales_delta

In [0]:
from delta import DeltaTable

dt = DeltaTable.forName(spark, "sales_delta")

display(dt.history())

In [0]:
DeltaTable.isDeltaTable(spark, "/FileStore/tables/data/input/userdata.parquet")

In [0]:
from delta import DeltaTable

DeltaTable.convertToDelta(
    spark,
    "parquet.`dbfs:/FileStore/tables/data/input/`"
)